# LAK-5 — Manifest explosion & rewrite

**Break → Detect → Fix → Prove.** The metadata cousin of LAK-2. Many small commits don't only
scatter *data* files — each commit also writes a new **manifest** (the metadata file listing data
files + their stats). A table with thousands of tiny manifests is slow to **plan** (the planner
reads manifests to prune files) even when the data itself is fine. The fix is **`rewrite_manifests`**.

**Pre-requisite:** the unified Spark server is up (`make up`) → Spark UI at http://localhost:4040.
This notebook connects via Spark Connect. **Laptop-safe:** a few rows per write across ~25 appends,
all under `.tmp/`; the table is dropped at the end (`make clean` clears the rest). The default
**tuned** box is fine — this is a metadata problem, not a memory one.

See the companion writeup in [`README.md`](./README.md) and the
[troubleshooting sheet](../../docs/troubleshooting.md) (LAK-5 row). Sibling: LAK-2 (small *data*
files) in [`../small_files/`](../small_files/) — both are metadata upkeep.

In [ ]:
from common.spark_session import spark, display_df
from common.iceberg_meta import table_health, compare_health
from common.metrics_diff import measure, compare
from pyspark.sql import functions as F

T = "iceberg_catalog.default.lak5_t"   # namespace.table lives under iceberg_catalog.default
N_APPENDS = 25                          # ~25 tiny commits → ~25 tiny manifests
spark

## Step 0 — create the table, then break it with many small commits

Each `append` is a separate **commit**: a new snapshot, and (roughly) a new **manifest**. We start
clean, then loop ~25 tiny appends — a few rows each. Nothing here is about volume; it's about
**metadata fragmentation**: one manifest per commit accumulates fast.

In [ ]:
spark.sql(f"DROP TABLE IF EXISTS {T}")

# Create the table from a first tiny batch (1 commit, 1 snapshot).
seed = (spark.range(0, 5).withColumnRenamed("id", "event_id")
        .withColumn("amount", F.round(F.rand(seed=0) * 100, 2)))
seed.writeTo(T).using("iceberg").create()
print("created", T, "with", spark.table(T).count(), "rows (1 commit so far)")

In [ ]:
# Break it: many small commits. Each append → its own snapshot → ~one new manifest.
for i in range(1, N_APPENDS + 1):
    batch = (spark.range(i * 5, i * 5 + 5).withColumnRenamed("id", "event_id")
             .withColumn("amount", F.round(F.rand(seed=i) * 100, 2)))
    batch.writeTo(T).append()

print(f"did {N_APPENDS} small appends; rows now: {spark.table(T).count():,} (still tiny)")

In [ ]:
# Headline metric for this module: manifests. Capture the 'before' health snapshot.
before = table_health(spark, T, "before rewrite")
print("manifests:", before["manifests"], "| snapshots:", before["snapshots"],
      "| data_files:", before["data_files"])

## 1. Detect it — read the metadata tables

This is a **metadata** pathology, so the tell is in Iceberg's own metadata tables, not the Spark UI
Stages tab. Count the manifests planning must read, and list them — each is tiny and references only
a file or two.

In [ ]:
mf = spark.sql(f"SELECT COUNT(*) AS n FROM {T}.manifests").first()["n"]
sn = spark.sql(f"SELECT COUNT(*) AS n FROM {T}.snapshots").first()["n"]
print(f"{T}.manifests : {mf}   <- planning opens & reads EVERY one of these to prune")
print(f"{T}.snapshots : {sn}   <- one per commit (≈ the manifest count)")

In [ ]:
# Each manifest is tiny: it references only a file or two (added_data_files_count).
display_df(spark.sql(f"""
    SELECT path, length, added_data_files_count, existing_data_files_count
    FROM {T}.manifests
    ORDER BY length
"""), limit=N_APPENDS + 5)

## 2. Diagnose

Every commit is atomic and self-contained — a writer never rewrites a previous commit's metadata, so
each append adds a **new manifest** (and a new manifest list for its snapshot). At plan time Iceberg
reads the current snapshot's manifest list, then opens **each manifest** to evaluate stats and prune
files. That cost is **per-manifest**, so it scales with the manifest *count*, independent of row
count or data size. A table with 25 rows and 26 manifests plans as if it were busy; more CPU won't
help — it's fixed metadata I/O.

In [ ]:
# (Optional) time a metadata query before the fix — planning reads every manifest.
# Over Spark Connect this captures wall-clock + stage metrics via the UI REST API.
m_before = measure(spark, "count (before rewrite)", lambda: spark.table(T).count())
print("runtime before:", m_before["runtime_s"], "s")

## 3. Fix it — `rewrite_manifests`

Coalesce many small manifests into a few larger ones in a single new snapshot. This rewrites only
**metadata** — the data files are untouched. The procedure returns how many manifests it
**rewrote** vs how many it **added** in their place (the second number is much smaller).

**Prevent it going forward:** keep `commit.manifest-merge.enabled` on (default `true`) so commits
merge into recent small manifests, and tune `commit.manifest.target-size-bytes` (default ~8 MB).
Schedule `rewrite_manifests` as routine maintenance, alongside `rewrite_data_files` (LAK-2).

In [ ]:
# table arg is 'namespace.table' — NO catalog prefix.
res = spark.sql(f"CALL iceberg_catalog.system.rewrite_manifests(table => 'default.lak5_t')")
res.show(truncate=False)   # rewritten_manifests_count / added_manifests_count

In [ ]:
after = table_health(spark, T, "after rewrite")
print("manifests:", after["manifests"], "| snapshots:", after["snapshots"],
      "| data_files:", after["data_files"], "(data files unchanged — metadata-only fix)")

## 4. Prove it

Before/after from `compare_health`. The **Manifests** row collapses while **Data files** stays flat
— the signature of a pure-metadata fix. (Snapshots tick up by one: the rewrite adds a snapshot; it
does **not** expire the old ones — that's LAK-3.) Optionally, the second `measure` shows the
metadata query getting cheaper now that planning reads far fewer manifests.

In [ ]:
compare_health([before, after])

# (Optional) planning-time after the fix — fewer manifests to read.
m_after = measure(spark, "count (after rewrite)", lambda: spark.table(T).count())
print()
compare([m_before, m_after])

## Takeaways & "in real production…"

- **Detect** manifest explosion from table **metadata** (`.manifests` count), not row counts — a
  tiny table can plan slowly with thousands of manifests.
- **Keep `commit.manifest-merge.enabled` on** and set a sensible `commit.manifest.target-size-bytes`
  so writers coalesce manifests as they go.
- **Schedule `rewrite_manifests`** — it's cheap (metadata only) and keeps planning fast.
- **LAK-2 vs LAK-5:** `rewrite_data_files` fixes too many small *data* files (slow **scans**);
  `rewrite_manifests` fixes too many small *manifests* (slow **planning**). Many-small-commits drives
  both, so production tables need both on a schedule — plus `expire_snapshots` (LAK-3) /
  `remove_orphan_files` (LAK-4) to reclaim what they leave behind.

## Teardown

In [ ]:
spark.sql(f"DROP TABLE IF EXISTS {T}")
print("Dropped", T, "— `make clean` clears all of .tmp if you want a fresh warehouse.")